# Soybean Disease Classification — Results & Figures

Companion to `train_all_models.py`. Reads everything that script wrote to
`RESULTS_ROOT` and produces, for the manuscript revision:

1. Backbone comparison table (mean ± std) + paste-ready LaTeX
2. Attention ablation table (mean ± std) + LaTeX
3. Statistical significance (Wilcoxon + paired *t*-test), SK vs. others
4. Training-accuracy / loss curves
5. Attention radar (spider) chart
6. Aggregated confusion matrix (proposed model)
7. Edge-deployment table (params / size / latency / FPS) + LaTeX
8. Cross-dataset external-validation tables + confusion matrices + LaTeX
9. Grad-CAM++ montages (ASDID + external field images)

**All figure text is set in Times New Roman** (falls back to the metric-compatible
Liberation Serif if Times New Roman is not installed — install `ttf-mscorefonts-installer`
on Linux for the real font).

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

# ----------------------------------------------------------------------
# Times New Roman setup (Liberation Serif fallback)
# ----------------------------------------------------------------------
def setup_times_new_roman():
    candidates = ['Times New Roman', 'TimesNewRoman', 'Times', 'Liberation Serif', 'DejaVu Serif']
    for f in fm.findSystemFonts():
        low = os.path.basename(f).lower()
        if 'times' in low or 'liberationserif' in low:
            try: fm.fontManager.addfont(f)
            except Exception: pass
    available = {f.name for f in fm.fontManager.ttflist}
    chosen = next((c for c in candidates if c in available), 'serif')
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': [chosen] + candidates,
        'mathtext.fontset': 'stix',
        'axes.unicode_minus': False,
        'font.size': 12, 'axes.linewidth': 1.0,
        'savefig.dpi': 300, 'figure.dpi': 110,
    })
    print('Font in use:', chosen)
    return chosen

setup_times_new_roman()

In [ ]:
# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
RESULTS_ROOT = '/home/bt/Desktop/Bee/soybean/results_consolidated'
FIG_DIR = os.path.join(RESULTS_ROOT, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)

# model key -> display name
DISPLAY = {
    'efficientnet_v2_s':  'EfficientNetV2-S',
    'mnasnet1_0':         'MNASNet',
    'mobilenet_v3_small': 'MobileNetV3-Small',
    'shufflenet_v2':      'ShuffleNet V2',
    'squeezenet':         'SqueezeNet',
    'effv2s_sknet':       'Proposed (SK)',
    'effv2s_dual':        'Dual (CBAM)',
    'effv2s_shuffle':     'Shuffle',
    'effv2s_simam':       'SimAM',
}
BACKBONE_ORDER  = ['efficientnet_v2_s','mnasnet1_0','mobilenet_v3_small','shufflenet_v2','squeezenet','effv2s_sknet']
ATTENTION_ORDER = ['effv2s_dual','effv2s_shuffle','effv2s_simam','effv2s_sknet']
PROPOSED = 'effv2s_sknet'

try:
    CLASS_NAMES = json.load(open(os.path.join(RESULTS_ROOT,'class_names.json')))['classes']
except Exception:
    CLASS_NAMES = ['Bacterial Blight','Cercospora Leaf Blight','Downy Mildew','Frogeye',
                   'Healthy','Potassium Deficiency','Soybean Rust','Target Spot']
print('Classes:', CLASS_NAMES)

def load_per_fold(key):
    return pd.read_csv(os.path.join(RESULTS_ROOT, key, 'per_fold_results.csv'))

In [ ]:
# ----------------------------------------------------------------------
# Helpers: mean ± std + LaTeX emitters
# ----------------------------------------------------------------------
METRICS = ['Accuracy','F1','Sensitivity','Precision','Specificity','MCC']

def ms(series, nd=4):
    return f"{series.mean():.{nd}f} $\\pm$ {series.std(ddof=1):.{nd}f}"

def summary_table(order):
    rows = []
    for key in order:
        df = load_per_fold(key)
        row = {'Model': DISPLAY[key]}
        for m in METRICS:
            row[m] = f"{df[m].mean():.4f}±{df[m].std(ddof=1):.4f}"
        row['Train time (s)'] = f"{df['TrainTime_s'].mean():.1f}"
        rows.append(row)
    return pd.DataFrame(rows)

def latex_table(order, metrics=METRICS, caption='', label=''):
    head = 'Model & ' + ' & '.join(metrics) + ' \\\\'
    lines = ['\\begin{table*}[ht]\\centering',
             f'\\caption{{{caption}}}', f'\\label{{{label}}}',
             '\\begin{tabular}{l' + 'c'*len(metrics) + '}', '\\toprule', head, '\\midrule']
    for key in order:
        df = load_per_fold(key)
        cells = [ms(df[m]) for m in metrics]
        name = DISPLAY[key]
        if key == PROPOSED:
            name = '\\textbf{'+name+'}'
            cells = ['\\textbf{'+c+'}' for c in cells]
        lines.append(name + ' & ' + ' & '.join(cells) + ' \\\\')
    lines += ['\\bottomrule','\\end{tabular}','\\end{table*}']
    return '\n'.join(lines)

## 1. Backbone comparison table

In [ ]:
tbl = summary_table(BACKBONE_ORDER); display(tbl)
print(latex_table(BACKBONE_ORDER,
      caption='Backbone comparison (mean $\\pm$ std across 10 folds).',
      label='tab:config_comparison'))

## 2. Attention ablation table

In [ ]:
tbl = summary_table(ATTENTION_ORDER); display(tbl)
print(latex_table(ATTENTION_ORDER, metrics=['Accuracy','Precision','Sensitivity','Specificity','F1'],
      caption='Attention-module ablation (mean $\\pm$ std across 10 folds).',
      label='tab:attention_ablation'))

## 3. Statistical significance

Paired tests across the 10 folds (same `random_state=42`, so folds are aligned).
Wilcoxon signed-rank is the primary test (no normality assumption); paired *t* is
reported alongside. Significant *p* < 0.05 means the proposed SK model's per-fold
scores differ systematically from the comparator.

In [ ]:
def paired_tests(key_a, key_b, metric='Accuracy'):
    a = load_per_fold(key_a)[metric].values
    b = load_per_fold(key_b)[metric].values
    try:    w_p = stats.wilcoxon(a, b).pvalue
    except Exception: w_p = float('nan')
    t_p = stats.ttest_rel(a, b).pvalue
    return a.mean()-b.mean(), w_p, t_p

print(f"Proposed (SK) vs. each comparator — metric: Accuracy\n{'-'*60}")
print(f"{'Comparator':22s} {'Δacc':>8s} {'Wilcoxon p':>12s} {'paired-t p':>12s}")
for key in ['efficientnet_v2_s','effv2s_dual','effv2s_shuffle','effv2s_simam']:
    d, wp, tp = paired_tests(PROPOSED, key, 'Accuracy')
    star = ' *' if (wp == wp and wp < 0.05) else ''
    print(f"{DISPLAY[key]:22s} {d:+8.4f} {wp:12.4f} {tp:12.4f}{star}")
print("\nPaste the SK-vs-EfficientNetV2-S and SK-vs-best-attention p-values into Section 'Statistical Significance'.")

## 4. Training curves (accuracy + loss)

In [ ]:
def fig_training_curves(order, out):
    fig, ax1 = plt.subplots(figsize=(11, 3.4)); ax2 = ax1.twinx()
    cmap = plt.cm.tab10(np.linspace(0, 1, len(order)))
    gmax = 1
    for c, key in zip(cmap, order):
        h = pd.read_csv(os.path.join(RESULTS_ROOT, key, 'training_history.csv'))
        g = h.groupby('Epoch').mean(numeric_only=True); gmax = g.index.max()
        ax1.plot(g.index, g['TrainAcc'], color=c, lw=2, label=DISPLAY[key])
        ax2.plot(g.index, g['TrainLoss'], color=c, lw=1.3, ls='--')
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Training Accuracy', fontweight='bold')
    ax2.set_ylabel('Training Loss', fontweight='bold')
    ax1.set_xlim(1, gmax); ax1.legend(loc='center right', fontsize=9, frameon=False)
    fig.tight_layout()
    for ext in ('png','pdf'): fig.savefig(out+'.'+ext, bbox_inches='tight')
    plt.show()

fig_training_curves(BACKBONE_ORDER, os.path.join(FIG_DIR, 'training_curves'))

## 5. Attention radar (spider) chart

In [ ]:
def fig_spider(order, metrics, out):
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist(); angles += angles[:1]
    fig, ax = plt.subplots(figsize=(6.2, 6.2), subplot_kw=dict(polar=True))
    for key in order:
        df = load_per_fold(key)
        vals = [df[m].mean() for m in metrics]; vals += vals[:1]
        ax.plot(angles, vals, lw=2, label=DISPLAY[key]); ax.fill(angles, vals, alpha=0.05)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics, fontweight='bold')
    ax.set_ylim(0.90, 1.0)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.10), frameon=False)
    fig.tight_layout()
    for ext in ('png','pdf'): fig.savefig(out+'.'+ext, bbox_inches='tight')
    plt.show()

fig_spider(ATTENTION_ORDER, ['Accuracy','Precision','Sensitivity','Specificity','F1'],
           os.path.join(FIG_DIR, 'spider'))

## 6. Confusion matrix (proposed model, aggregated out-of-fold)

In [ ]:
from sklearn.metrics import confusion_matrix
def fig_confusion(y_true, y_pred, names, out, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(names)))
    M = cm.astype(float)/cm.sum(axis=1, keepdims=True) if normalize else cm
    fig, ax = plt.subplots(figsize=(7.6, 6.6))
    im = ax.imshow(M, cmap='Oranges')
    ax.set_xticks(range(len(names))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right'); ax.set_yticklabels(names)
    thr = M.max()/2
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = f"{M[i,j]:.2f}" if normalize else f"{int(M[i,j])}"
            ax.text(j, i, v, ha='center', va='center',
                    color='white' if M[i,j] > thr else 'black', fontsize=9)
    ax.set_xlabel('Predicted Labels', fontweight='bold'); ax.set_ylabel('True Labels', fontweight='bold')
    fig.tight_layout()
    for ext in ('png','pdf'): fig.savefig(out+'.'+ext, bbox_inches='tight')
    plt.show()

d = np.load(os.path.join(RESULTS_ROOT, PROPOSED, 'oof_predictions.npz'))
fig_confusion(d['y_true'], d['y_pred'], CLASS_NAMES, os.path.join(FIG_DIR, 'confusion'))

## 7. Edge-deployment table

In [ ]:
rows = []
for key in BACKBONE_ORDER:
    p = os.path.join(RESULTS_ROOT, key, 'edge_profile.json')
    if not os.path.exists(p): continue
    e = json.load(open(p))
    rows.append({'Model': DISPLAY[key], 'Params (M)': e['parameters_millions'],
                 'ONNX size (MB)': e.get('onnx_size_mb'),
                 'CPU latency (ms)': e['cpu_latency_ms'], 'CPU FPS': e['cpu_fps'],
                 'GPU FPS': e.get('gpu_fps')})
edge = pd.DataFrame(rows); display(edge)
print('% LaTeX rows for the edge table:')
for _, r in edge.iterrows():
    print(f"{r['Model']} & {r['Params (M)']} & {r['ONNX size (MB)']} & "
          f"{r['CPU latency (ms)']} & {r['CPU FPS']} & {r['GPU FPS']} \\\\")

## 8. Cross-dataset external validation

Reads `cross_dataset/cross_dataset_metrics.json` and `predictions.npz`.

In [ ]:
cd_dir = os.path.join(RESULTS_ROOT, 'cross_dataset')
cd = json.load(open(os.path.join(cd_dir, 'cross_dataset_metrics.json')))
print(f"External test images (shared classes): {cd['n_test_images']}")
overall = pd.DataFrame([
    {'Protocol':'Open (8-way head)', **{k:round(cd['open'][k],4) for k in
        ['accuracy','precision_macro','sensitivity_macro','f1_macro']}},
    {'Protocol':'Constrained (4 classes)', **{k:round(cd['constrained'][k],4) for k in
        ['accuracy','precision_macro','sensitivity_macro','f1_macro']}},
])
display(overall)
pc = pd.DataFrame([{'Class':c, 'Precision':round(v['precision'],4),
                    'Recall':round(v['recall'],4), 'Support':v['support']}
                   for c,v in cd['constrained']['per_class'].items()])
display(pc)

# confusion matrices for both protocols (over shared classes)
pred = np.load(os.path.join(cd_dir, 'predictions.npz'))
shared = list(pred['shared_idx']); shared_names = [CLASS_NAMES[i] for i in shared]
remap = {orig:i for i,orig in enumerate(shared)}
yt = np.array([remap[v] for v in pred['y_true']])
for proto, arr in [('open', pred['y_pred_open']), ('constrained', pred['y_pred_constrained'])]:
    yp = np.array([remap.get(int(v), -1) for v in arr])  # -1 = predicted a non-shared class
    names = shared_names + (['(other)'] if (yp==-1).any() else [])
    yp = np.where(yp==-1, len(shared_names), yp)
    fig_confusion(yt, yp, names, os.path.join(FIG_DIR, f'cross_{proto}'), normalize=False)

## 9. Grad-CAM++ montages (ASDID + external field images)

Requires torch + the trained proposed checkpoint. Produces the `gradcam.png`
and the new `gradcam_external.png` figure referenced in the manuscript.

In [ ]:
import torch, torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import importlib.util

# import the model + transforms from the training script
spec = importlib.util.spec_from_file_location('tam', 'train_all_models.py')
tam = importlib.util.module_from_spec(spec); spec.loader.exec_module(tam)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = tam.MODEL_REGISTRY[PROPOSED][0]().to(device)
model.load_state_dict(torch.load(os.path.join(RESULTS_ROOT, PROPOSED, f'{PROPOSED}_best_overall.pth'),
                                 map_location=device))
model.eval()
target_layer = model.features[-1]   # last conv block of EfficientNetV2-S

class GradCAMpp:
    def __init__(self, model, layer):
        self.model = model; self.acts = None; self.grads = None
        layer.register_forward_hook(lambda m,i,o: setattr(self, 'acts', o))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self, 'grads', go[0]))
    def __call__(self, x, cls=None):
        out = self.model(x); cls = out.argmax(1).item() if cls is None else cls
        self.model.zero_grad(); out[0, cls].backward()
        g, a = self.grads[0], self.acts[0]
        g2, g3 = g.pow(2), g.pow(3)
        denom = 2*g2 + (a*g3).sum(dim=(1,2), keepdim=True)
        alpha = g2 / (denom + 1e-7)
        w = (alpha * F.relu(g)).sum(dim=(1,2))
        cam = F.relu((w[:,None,None]*a).sum(0))
        cam = (cam - cam.min())/(cam.max()-cam.min()+1e-7)
        return cam.cpu().numpy(), cls

prep = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
                           transforms.Normalize(tam.IMAGENET_MEAN, tam.IMAGENET_STD)])

def overlay(img_path, cam):
    import matplotlib.cm as cm
    raw = Image.open(img_path).convert('RGB').resize((224,224))
    import numpy as np
    heat = cm.jet(np.array(Image.fromarray((cam*255).astype('uint8')).resize((224,224)))/255.)[...,:3]
    return (0.5*np.array(raw)/255. + 0.5*heat)

def montage(image_paths, titles, out):
    cammer = GradCAMpp(model, target_layer)
    n = len(image_paths)
    fig, axes = plt.subplots(2, n, figsize=(2.2*n, 4.6))
    for k, (p, t) in enumerate(zip(image_paths, titles)):
        x = prep(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
        cam, _ = cammer(x)
        axes[0,k].imshow(Image.open(p).convert('RGB').resize((224,224))); axes[0,k].axis('off')
        axes[1,k].imshow(overlay(p, cam)); axes[1,k].axis('off')
        axes[1,k].set_title(t, fontsize=11, fontweight='bold')
    axes[0,0].set_ylabel('Raw', fontweight='bold'); axes[1,0].set_ylabel('Grad-CAM++', fontweight='bold')
    fig.tight_layout()
    for ext in ('png','pdf'): fig.savefig(out+'.'+ext, bbox_inches='tight')
    plt.show()

# EDIT these to point at one representative image per class (ASDID + external).
# Example:
# asdid_imgs = ['/path/Bacterial blight/x.jpg', ...]; montage(asdid_imgs, CLASS_NAMES, os.path.join(FIG_DIR,'gradcam'))
# ext_imgs   = ['/path/Bacterial Blight/y.jpg', ...]; montage(ext_imgs, ['Bacterial blight','Cercospora','Healthy','Rust'], os.path.join(FIG_DIR,'gradcam_external'))
print('Grad-CAM++ ready. Set image paths above and call montage(...).')

## (Optional) LIME and SHAP

If you want to regenerate the `merge.png` LIME/SHAP figure, install `lime` and
`shap`. Use `prep` + `model` above as the classifier. Kept optional to avoid
heavy dependencies in this notebook.